# Compliance (TICK_AND_FLICK) — end-to-end QC

Run the `compliance.tick_and_flick_wow` and `compliance.tick_and_flick_vs_chain`
detectors through the full pipeline: detect → score → facts → investigate.

This notebook loads both the LSM_ALERT_FEEDBACK table (used by the compliance
detector) and the OSA table (used to compute the projected cost of dismissed
alerts), then extracts the first `N_SAMPLES` raw `signal_json` and `facts_json`
pairs to disk without writing to the alerts table.

In [ ]:
# --- QC parameters ------------------------------------------------------------
ANALYSIS_DATE_STR = '2026-07-26'   # last day of the week
CLIENT = 'eastofengland'            # or None for base defaults
QC_STORE_IDS = [155]                # set to None for all stores
N_SAMPLES = 2                       # how many signal_json / facts_json pairs to save
OUTPUT_DIR = 'qc_outputs'           # directory for saved JSON files

import datetime
analysis_date = datetime.date.fromisoformat(ANALYSIS_DATE_STR)
THIS_WEEK_END = str(analysis_date)
THIS_WEEK_START = str(analysis_date - datetime.timedelta(days=6))
LAST_WEEK_END = str(analysis_date - datetime.timedelta(days=7))
LAST_WEEK_START = str(analysis_date - datetime.timedelta(days=13))

print(f'Analysis date : {THIS_WEEK_END}')
print(f'This week   : {THIS_WEEK_START} to {THIS_WEEK_END}')
print(f'Last week   : {LAST_WEEK_START} to {LAST_WEEK_END}')

In [ ]:
import sys, os, logging, warnings

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

logging.getLogger('snowflake.connector').setLevel(logging.ERROR)
logging.getLogger('snowflake.snowpark').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='snowflake')

from store_insight.pipeline.config.loader import load_config
from store_insight.pipeline.pipeline.pipeline import run_pipeline, derive_windows
from store_insight.pipeline.pipeline.session import create_session
from snowflake.snowpark import functions as F

cfg = load_config(CLIENT)
session = create_session(cfg)

# Restrict to the compliance detector only
cfg.detectors.enabled = ['compliance.tick_and_flick_wow', 'compliance.tick_and_flick_vs_chain']

windows = derive_windows(THIS_WEEK_END, cfg=cfg)
LOOKBACK_START = windows['lookback_start']

print(f'Config loaded: client={cfg.client_name}')
print(f'Enabled detectors: {cfg.detectors.enabled}')

In [ ]:
# --- Load OSA data (needed for projected-cost scoring) ------------------------
OSA_TABLE = cfg.tables.osa
HIERARCHY_TABLE = cfg.tables.hierarchy

osa_raw = (
    session.table(OSA_TABLE)
    .filter(F.col('DATE') >= LOOKBACK_START)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .select(
        F.col('STORE_ID'),
        F.col('ITEM_CODE'),
        F.col('ACTUAL_SALES_VOLUME'),
        F.col('LOST_SALES_VOLUME'),
        F.col('LOST_SALES_VALUE'),
        F.col('DATE'),
        F.col('INVENTORY_QTY'),
        F.col('DEPARTMENT_ID'),
        F.col('REPORTING_PERIOD_ID'),
    )
)

item = (
    session.table(HIERARCHY_TABLE)
    .select(
        F.col('ITEM_CODE'),
        F.col('ITEM_DESCRIPTION'),
        F.col('ITEM_HIERARCHY_1'),
        F.col('ITEM_HIERARCHY_2'),
        F.col('ITEM_HIERARCHY_3'),
    )
    .distinct()
)

osa = (
    osa_raw
    .join(item, on=['ITEM_CODE'], how='left')
    .filter(F.col('ITEM_HIERARCHY_2').is_not_null())
    .cache_result()
)

print(f'OSA rows loaded & cached: {osa.count():,}')

In [ ]:
# --- Load LSM_ALERT_FEEDBACK data (required by TICK_AND_FLICK) ----------------
FEEDBACK_TABLE = cfg.tables.feedback

feedback = (
    session.table(FEEDBACK_TABLE)
    .filter(F.col('DATE') >= LAST_WEEK_START)
    .filter(F.col('DATE') <= THIS_WEEK_END)
    .select(
        F.col('STORE_ID'),
        F.col('SITE_NAME'),
        F.col('DATE'),
        F.col('ALERT_SOURCE_ID'),
        F.col('INTERVENTION_ID'),
        F.col('TICK_AND_FLICK_FLAG'),
        F.col('ALERT_TYPE_NAME'),
        F.col('ITEM_CODE'),
    )
    .cache_result()
)

print(f'Feedback rows loaded & cached: {feedback.count():,}')

In [ ]:
# Run detect → score → facts → investigate, but do NOT write to the output table.
cards = run_pipeline(
    osa,
    last_week_start=LAST_WEEK_START,
    last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START,
    this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    store_id=QC_STORE_IDS,
    session=session,
    run_date=THIS_WEEK_END,
    feedback=feedback,
    cfg=cfg,
)

print(f'\nTotal cards produced: {len(cards)}')

In [ ]:
import json
from pathlib import Path

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

if not cards:
    print('No cards produced; nothing to save.')
else:
    n = min(N_SAMPLES, len(cards))
    print(f'Saving signal_json and facts_json for the first {n} card(s) to {out_dir.resolve()}:\n')
    for i, card in enumerate(cards[:n], start=1):
        signal_path = out_dir / f'compliance_signal_{i:03d}.json'
        facts_path = out_dir / f'compliance_facts_{i:03d}.json'
        signal_path.write_text(json.dumps(card['signal_json'], indent=2, default=str))
        facts_path.write_text(json.dumps(card['facts_json'], indent=2, default=str))
        print(f'Card {i}: store_id={card["store_id"]} section={card["section"]}')
        print(f'  signal_json -> {signal_path}')
        print(f'  facts_json  -> {facts_path}')